In [1]:
import nltk
import codecs
from nltk import *
from nltk.draw.tree import draw_trees
import pymorphy2 as pm

In [2]:
## загружаем PyMorphy2
m = pm.MorphAnalyzer()
## открываем (создаем)файл с грамматикой, куда будут записываться правила
f = codecs.open(r"C:\Users\Андрей\AppData\Roaming\nltk_data\grammars\book_grammars\test.fcfg", mode= "w", encoding = "utf-8")
rules = codecs.open("rules.txt", mode= "r", encoding = "utf-8")
## записываем правила, которые вручную делаем (некоторые на основе правил из АОТ)
for rule in rules:
	f.writelines(rules)

In [3]:
rules.close()
f.close()

In [4]:
## функция, которая переводит нужную нам информацию из пайморфи в вид, читаемый парсером NLTK
## принимает (токенизированное) словосочетание на входе, записывает правила (lexical productions) в тот же файл с грамматикой

In [5]:
def pm2fcfg(phrase): ## phrase - это словосочетание, которое мы разбираем
    f = codecs.open(r"C:\Users\Андрей\AppData\Roaming\nltk_data\grammars\book_grammars\test.fcfg", mode= "a", encoding = "utf-8")
    for x in phrase:
        a = m.parse(x) ## a - список возможных вариантов морфологического разбора слова, предлагаемых пайморфи
		## от части речи зависит, какие признаки отправляются в грамматику, осюда условия
        for y in a:
            if (y.tag.POS == "NOUN") or (y.tag.POS == "ADJF") or (y.tag.POS == "PRTF"):
                strk = str(y.tag.POS) + "[C=" + str(y.tag.case) + ", G=" + str(y.tag.gender) + ", NUM=" + str(y.tag.number) + ", PER=3" + ", NF=u'" + str(y.normal_form) + "'] -> '" + str(y.word) + "'\n"
                f.writelines(strk)
            elif (y.tag.POS == "ADJS") or (y.tag.POS == "PRTS"):
                strk = str(y.tag.POS) + "[G=" + str(y.tag.gender) + ", NUM=" + str(y.tag.number) + ", NF=u'" + str(y.normal_form) + "'] -> '" + str(y.word) + "'\n"
                f.writelines(strk)
            elif (y.tag.POS == "NUMR"):
                strk = str(y.tag.POS) + "[C=" + str(y.tag.case) + ", NF=u'" + str(y.normal_form) + "'] -> '" + str(y.word) + "'\n"
                f.writelines(strk)
            elif (y.tag.POS == "ADVB") or (y.tag.POS == "GRND") or (y.tag.POS == "COMP") or (y.tag.POS == "PRED") or (y.tag.POS == "PRCL") or (y.tag.POS == "INTJ"):
                strk = str(y.tag.POS) + "[NF=u'" + str(y.normal_form) + "'] -> '" + str(y.word) + "'\n"
                f.writelines(strk)
            elif (y.tag.POS == "PREP") or (y.tag.POS == "CONJ"):
                strk = str(y.tag.POS) + "[NF=u'" + str(y.normal_form) + "'] -> '" + str(y.word) + "'\n"
                f.writelines(strk)
                break
            elif (y.tag.POS == "NPRO") & (y.normal_form != "это") & (y.normal_form != "нечего") & (y.normal_form != 'кто'):
                if ((y.tag.person[0] == "3") & (y.tag.number == "sing")):                    
                    strk = str(y.tag.POS) + "[C=" + str(y.tag.case) + ", G=" + str(y.tag.gender) + ", NUM=" + str(y.tag.number) + ", PER=" + str(y.tag.person)[0] + ", NF=u'" + str(y.normal_form) + "'] -> '" + str(y.word) + "'\n"
                else:
                    strk = str(y.tag.POS) + "[C=" + str(y.tag.case) + ", NUM=" + str(y.tag.number) + ", PER=" + str(y.tag.person)[0] + ", NF=u'" + str(y.normal_form) + "'] -> '" + str(y.word) + "'\n"
                f.writelines(strk)
            elif (y.tag.POS == "NPRO") & (y.normal_form == 'кто'):
                strk = str(y.tag.POS) + "[C=" + str(y.tag.case) + ", NUM=" + str(y.tag.number) + ", NF=u'" + str(y.normal_form) + "'] -> '" + str(y.word) + "'\n"
                f.writelines(strk)
            elif (y.tag.POS == "VERB")  or (y.tag.POS == "INFN"):
                if (y.tag.tense == "past"):                    
                    strk = str(y.tag.POS) + "[TR=" + str(y.tag.transitivity) + ", TENSE=" + str(y.tag.tense) + ", G=" + str(y.tag.gender) + ", NUM=" + str(y.tag.number) + ", PER=" + "0" + ", NF=u'" + str(y.normal_form) + "'] -> '" + str(y.word) + "'\n"
                elif (y.tag.POS == "INFN"):
                    strk = str(y.tag.POS) + "[TR=" + str(y.tag.transitivity) + ", TENSE=0, G=0, NUM=0, PER=0, NF=u'" + str(y.normal_form) + "'] -> '" + str(y.word) + "'\n"
                else:
                    strk = str(y.tag.POS) + "[TR=" + str(y.tag.transitivity) + ", TENSE=" + str(y.tag.tense) + ", G=" + "0" + ", NUM=" + str(y.tag.number) + ", PER=" + str(y.tag.person)[0] + ", NF=u'" + str(y.normal_form) + "'] -> '" + str(y.word) + "'\n"
                f.writelines(strk)
    f.close()

In [7]:
text = 'ни к кому обратиться' ## сюда пишется словосочетание для разбора
words = nltk.word_tokenize(text.lower()) ## разбиваем словосочетание на токены

In [8]:
pm2fcfg(words) ## запускаем функцию, описанную выше
cp = load_parser('grammars/book_grammars/test.fcfg') ## открываем нашу грамматику, смотрим на разбор в консоли или ещё где

In [9]:
print(cp.grammar())

Grammar with 104 productions (start state = XP[])
    XP[] -> NP[]
    XP[] -> VP[]
    XP[] -> AdjP[]
    XP[] -> AdvP[]
    XP[] -> COMP[]
    XP[] -> PP[]
    XP[] -> NUMRNP[]
    XP[] -> NUMR[]
    XP[] -> S[]
    XP[] -> CONJP[]
    XP[] -> PREDP[]
    XP[] -> PrtfP[]
    S[-inv] -> NP[C='nomn', G=?g, NUM=?n, PER=?p] VP[G=?g, NUM=?n, PER=0]
    S[-inv] -> NP[C='nomn', G=?g, NUM=?n, PER=?p] VP[G=0, NUM=?n, PER=?p]
    S[-inv] -> AdjP[C='nomn', G=?g, NUM=?n, PER=?p] VP[G=?g, NUM=?n, PER=0]
    S[-inv] -> AdjP[C='nomn', G=?g, NUM=?n, PER=?p] VP[G=0, NUM=?n, PER=?p]
    S[-inv] -> NP[C='nomn', NUM='plur', PER=?p] VP[G=None, NUM=?n, PER=0]
    S[+adj] -> NP[C='nomn', G=?g, NUM=?n, PER=?p] AdjP[C='nomn', G=?g, NUM=?n, PER=?p]
    S[-inv] -> NUMRNP[C='nomn'] VP[]
    S[+inv] -> VP[G=?g, NUM=?n, PER=0] NP[C='nomn', G=?g, NUM=?n, PER=?p]
    S[+inv] -> VP[G=0, NUM=?n, PER=?p] NP[C='nomn', G=?g, NUM=?n, PER=?p]
    S[+advp] -> AdvP[] S[+advp]
    S[+advp] -> NP[C='gent', G=?g, NUM=?n, PER=?

In [10]:
output =  cp.parse(words)
for tree in output:
		print(tree)

(XP[]
  (VP[G=0, NUM=0, PER=0, TENSE=0]
    (CONJP[+neg]
      ни
      (PP[C=?c, G='masc', NUM='sing', PER=3]
        (PREP[NF='к'] к)
        (NP[C='datv', G='masc', NUM='sing', PER=3]
          (NOUN[C='datv', G='masc', NF='ком', NUM='sing', PER=3]
            кому))))
    (VP[G=0, NUM=0, PER=0, TENSE=0]
      (INFN[G=0, NF='обратиться', NUM=0, PER=0, TENSE=0, TR='intr']
        обратиться))))
(XP[]
  (VP[G=0, NUM=0, PER=0, TENSE=0]
    (CONJP[+neg]
      ни
      (PP[C=?c, G='femn', NUM='sing', PER=3]
        (PREP[NF='к'] к)
        (NP[C='accs', G='femn', NUM='sing', PER=3]
          (NOUN[C='accs', G='femn', NF='кома', NUM='sing', PER=3]
            кому))))
    (VP[G=0, NUM=0, PER=0, TENSE=0]
      (INFN[G=0, NF='обратиться', NUM=0, PER=0, TENSE=0, TR='intr']
        обратиться))))
(XP[]
  (VP[G=0, NUM=0, PER=0, TENSE=0]
    (CONJP[+neg]
      ни
      (PP[C=?c, G=?g, NUM='sing', PER=?p]
        (PREP[NF='к'] к)
        (NP[C='datv', G=?g, NUM='sing', PER=?p]
          (NPRO[C=

In [11]:
list(output)

[]

In [12]:
draw_trees(*(tree for tree in cp.parse(words)))